In [1]:
import requests
from bs4 import BeautifulSoup
import csv
import time
from urllib.parse import quote
from datetime import datetime, timedelta

In [ ]:
# --- MANUAL CONFIGURATION ---
# Set your exact search query here. 
# The space acts as an AND operator in DCInside search.
COMPOUND_KEYWORD = "임현식" 

# Set the target date based on the article's publication
MIDDLE_DATE_STR = "2025-03-24-22-53"  # YYYY-MM-DD-HH-MM
middle_dt = datetime.strptime(MIDDLE_DATE_STR, "%Y-%m-%d-%H-%M")

# Time window calculation
calculated_start_dt = middle_dt - timedelta(days=15)
calculated_end_dt = middle_dt + timedelta(days=60)

START_DATE_STR = calculated_start_dt.strftime("%Y-%m-%d-%H-%M")
END_DATE_STR = calculated_end_dt.strftime("%Y-%m-%d-%H-%M")

# Update output filename to use the compound keyword, replacing spaces with underscores
safe_filename_keyword = COMPOUND_KEYWORD.replace(" ", "_")
OUTPUT_FILE = f"3E_{safe_filename_keyword}_list.csv"

print(f"Target Query: '{COMPOUND_KEYWORD}'")
print(f"Time Window: {START_DATE_STR} to {END_DATE_STR}")
print(f"Output File: {OUTPUT_FILE}")

Target Query: '김재중 할아버지'
Time Window: 2025-03-09-22-53 to 2025-05-23-22-53
Output File: 3E_김재중_할아버지_list.csv


In [15]:
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

def get_custom_encoded_keyword(keyword):
    """
    Converts '김종국 결혼식' -> '.EA.B9.80.EC.A2.85.EA.B5.AD.20.EA.B2.B0.ED.98.BC.EC.8B.9D'
    Standard URL encode uses %, DC Inside uses . for paths.
    """
    encoded = quote(keyword.encode('utf-8')).replace('%', '.')
    return encoded

def parse_date_arg(date_str):
    """Parses input string 'YYYY-MM-DD-HH-MM' to datetime object."""
    return datetime.strptime(date_str, "%Y-%m-%d-%H-%M")

def parse_article_date(date_text):
    """Parses article date '2025.12.28 10:41' to datetime object."""
    return datetime.strptime(date_text, "%Y.%m.%d %H:%M")

In [16]:
def scrape_dcinside():
    start_dt = parse_date_arg(START_DATE_STR)
    end_dt = parse_date_arg(END_DATE_STR)
    
    encoded_keyword = get_custom_encoded_keyword(COMPOUND_KEYWORD)
    
    with open(OUTPUT_FILE, mode='w', newline='', encoding='utf-8-sig') as f:
        writer = csv.writer(f)
        writer.writerow(['keyword', 'time', 'title', 'URL'])
        
        page = 1
        stop_scraping = False
        
        while not stop_scraping:
            url = f"https://search.dcinside.com/post/p/{page}/sort/latest/q/{encoded_keyword}"
            print(f"Scanning Page {page}: {url}")
            
            try:
                response = requests.get(url, headers=HEADERS)
                if response.status_code != 200:
                    print(f"Error: Status code {response.status_code}")
                    break
                
                soup = BeautifulSoup(response.text, 'html.parser')
                
                result_list = soup.select('ul.sch_result_list > li')
                
                if not result_list:
                    print("No more results found.")
                    break
                
                for li in result_list:
                    date_element = li.select_one('.link_dsc_txt.dsc_sub .date_time')
                    if not date_element:
                        continue
                        
                    article_date_str = date_element.get_text(strip=True)
                    try:
                        article_dt = parse_article_date(article_date_str)
                    except ValueError:
                        print(f"Skipping date parse error: {article_date_str}")
                        continue

                    if article_dt > end_dt:
                        continue
                        
                    if article_dt < start_dt:
                        print(f"Reached date {article_dt}, which is older than start time. Stopping.")
                        stop_scraping = True
                        break
                    
                    link_tag = li.select_one('a.tit_txt')
                    if link_tag:
                        title = link_tag.get_text(strip=True)
                        article_url = link_tag['href']
                        
                        writer.writerow([COMPOUND_KEYWORD, article_date_str, title, article_url])
            
            except Exception as e:
                print(f"An error occurred: {e}")
                break
            
            page += 1

            if page > 120:
                print("Reached the 120-page limit.")
                break
            
            time.sleep(0.5)

    print(f"Scraping finished. Data saved to {OUTPUT_FILE}")

if __name__ == "__main__":
    scrape_dcinside()

Scanning Page 1: https://search.dcinside.com/post/p/1/sort/latest/q/.EA.B9.80.EC.9E.AC.EC.A4.91.20.ED.95.A0.EC.95.84.EB.B2.84.EC.A7.80
Reached date 2024-10-24 20:07:00, which is older than start time. Stopping.
Scraping finished. Data saved to 3E_김재중_할아버지_list.csv
